In [15]:
import time
import os
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

URL = "https://iuh.edu.vn/vi/hop-tac.html/p={p}"
LABEL = "Hợp tác"
CSV_FILE = "hop_tac.csv"

# ✅ đợi đúng link bài viết (h3 a) có href .html
ARTICLE_LINK_CSS = "h3 a[href$='.html']"

def make_driver(headless=True):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,900")
    return webdriver.Chrome(options=options)

def scroll_bottom(driver, rounds=3, pause=0.6):
    for _ in range(rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

driver = make_driver(headless=True)

file_exists = os.path.exists(CSV_FILE)
seen_href = set()  # ✅ chống trùng + ổn định hơn seen theo text

try:
    with open(CSV_FILE, mode="a", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(["STT", "Thông tin", "Label"])
            stt = 1
        else:
            with open(CSV_FILE, encoding="utf-8-sig") as rf:
                stt = sum(1 for _ in rf)  # gồm header

        for p in range(0, 8):
            driver.get(URL.format(p=p))

            # ✅ đợi trang load xong
            WebDriverWait(driver, 25).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )

            # ✅ đợi đúng link bài viết xuất hiện
            WebDriverWait(driver, 25).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ARTICLE_LINK_CSS))
            )

            # ✅ scroll để tránh lazy-load miss
            scroll_bottom(driver, rounds=3, pause=0.5)

            links = driver.find_elements(By.CSS_SELECTOR, ARTICLE_LINK_CSS)

            added = 0
            for a in links:
                href = a.get_attribute("href") or ""
                if not href or href in seen_href:
                    continue

                text = a.text.strip()
                if not text:
                    continue

                seen_href.add(href)
                writer.writerow([stt, text, LABEL])
                f.flush()
                print(f"✔ p={p} wrote row {stt}")
                stt += 1
                added += 1

                time.sleep(1)

            print(f"✅ page {p} done | links={len(links)} | added={added}")
            time.sleep(1)

finally:
    driver.quit()

print("🎉 DONE – CSV được ghi từng dòng")


✔ p=0 wrote row 1
✔ p=0 wrote row 2
✔ p=0 wrote row 3
✔ p=0 wrote row 4
✔ p=0 wrote row 5
✔ p=0 wrote row 6
✔ p=0 wrote row 7
✔ p=0 wrote row 8
✔ p=0 wrote row 9
✔ p=0 wrote row 10
✔ p=0 wrote row 11
✔ p=0 wrote row 12
✔ p=0 wrote row 13
✔ p=0 wrote row 14
✔ p=0 wrote row 15
✅ page 0 done | links=15 | added=15
✔ p=1 wrote row 16
✔ p=1 wrote row 17
✔ p=1 wrote row 18
✔ p=1 wrote row 19
✔ p=1 wrote row 20
✔ p=1 wrote row 21
✔ p=1 wrote row 22
✔ p=1 wrote row 23
✔ p=1 wrote row 24
✔ p=1 wrote row 25
✔ p=1 wrote row 26
✔ p=1 wrote row 27
✔ p=1 wrote row 28
✔ p=1 wrote row 29
✔ p=1 wrote row 30
✅ page 1 done | links=15 | added=15
✔ p=2 wrote row 31
✔ p=2 wrote row 32
✔ p=2 wrote row 33
✔ p=2 wrote row 34
✔ p=2 wrote row 35
✔ p=2 wrote row 36
✔ p=2 wrote row 37
✔ p=2 wrote row 38
✔ p=2 wrote row 39
✔ p=2 wrote row 40
✔ p=2 wrote row 41
✔ p=2 wrote row 42
✔ p=2 wrote row 43
✔ p=2 wrote row 44
✔ p=2 wrote row 45
✅ page 2 done | links=15 | added=15
✔ p=3 wrote row 46
✔ p=3 wrote row 47
✔ p=3 wr